# Validation 11 — CSV Export (Cell 30)
Verifies all 4 CSV files are written, readable, and row counts match.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, warnings
from pathlib import Path
from scipy.signal import butter,cheby1,cheby2,ellip,bessel,sosfiltfilt,sosfreqz,welch
from scipy.signal import spectrogram as sp_spectrogram
from scipy.io import wavfile
from itertools import product
warnings.filterwarnings('ignore')

BASE_DIR    = Path(r"D:\\1 placement\\IAESTE INTERNSHIP CZECH\\iaeste26-blasting-sound-main\\iaeste26-blasting-sound-main")
DATA_DIR    = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_PRIORITY   = ['AccAxial4507','AccRadial4507','Mic147EB','Mic46BE']
ENERGY_WINDOW_S   = 0.05;  NOISE_DURATION_S = 0.5;  ONSET_THRESHOLD = 10.0;  ONSET_OFFSET_S = 7.0
WINDOW_DURATION_S = 5.0;   WINDOW_STEP_S    = 1.0
LOWER_LIMITS = [176,225,283,353,440,565,707,880,1130,1414,1760,10,10,500,1000]
UPPER_LIMITS = [225,283,353,440,565,707,880,1130,1414,1760,2220,1000,2000,1500,2000]
N_BANDS      = len(LOWER_LIMITS)
BAND_LABELS  = [f"{lo}–{hi} Hz" for lo,hi in zip(LOWER_LIMITS,UPPER_LIMITS)]
FILTER_TYPES  = ['Butterworth','Chebyshev I','Chebyshev II','Elliptical','Bessel']
FILTER_ORDERS = [3,5,7]
CHEBY1_RIPPLE_DB=0.5; CHEBY2_ATTEN_DB=40.0; ELLIP_RIPPLE_DB=0.5; ELLIP_ATTEN_DB=40.0

P=[0]; F=[0]
def check(label, ok, note=""):
    s="[PASS]" if ok else "[FAIL]"
    if ok: P[0]+=1
    else:  F[0]+=1
    print(f"  {s}  {label}" + (f"  → {note}" if note else ""))
def info(label, val): print(f"  [INFO]  {label}: {val}")
def summary():
    t=P[0]+F[0]
    print(f"\n{'='*50}")
    print(f"  PASS: {P[0]}/{t}  |  FAIL: {F[0]}/{t}")
    print(f"  Score: {P[0]/t*100:.0f}%" if t else "  No checks run")
    print('='*50)
print("Config loaded.")


Config loaded.


In [2]:
csv_main=RESULTS_DIR/"blasting_all_results.csv"
csv_band=RESULTS_DIR/"blasting_band_averages.csv"
csv_cx  =RESULTS_DIR/"filter_complexity.csv"
csv_sum =RESULTS_DIR/"blasting_file_summary.csv"

check("blasting_all_results.csv exists",    csv_main.exists(), str(csv_main) if csv_main.exists() else "NOT FOUND — run validation_08 first")
check("filter_complexity.csv exists",       csv_cx.exists(),   str(csv_cx)   if csv_cx.exists()   else "NOT FOUND — run validation_10 first")

if csv_main.exists():
    df=pd.read_csv(csv_main)
    # Regenerate band averages and file summary
    df_band=(df.groupby(['filename','sensor','filter_type','order','band_idx','band_label'])
               [['rms','peak','crest_factor','zcr','band_power','spectral_centroid']]
               .mean().round(6).reset_index())
    df_band.to_csv(csv_band,index=False)
    df_fs=(df.groupby(['filename','sensor','filter_type','order'])
              [['rms','peak','crest_factor','zcr']].agg(['mean','std']).round(6).reset_index())
    df_fs.columns=['_'.join(c).strip('_') for c in df_fs.columns]
    df_fs.to_csv(csv_sum,index=False)

    check("blasting_band_averages.csv written",  csv_band.exists(), f"{csv_band.stat().st_size:,} bytes")
    check("blasting_file_summary.csv written",   csv_sum.exists(),  f"{csv_sum.stat().st_size:,} bytes")

    df2=pd.read_csv(csv_main)
    check("Reloaded row count matches original", len(df2)==len(df), f"{len(df2)} rows")
    check("All parameter columns present after reload",
          all(c in df2.columns for c in ['rms','peak','crest_factor','zcr','band_power','spectral_centroid']))
    check("No NaN in RMS after reload",          df2['rms'].notna().all())
    check("filename column present",             'filename' in df2.columns)
    check("sensor column present",               'sensor'   in df2.columns)

    info("Full results rows",  f"{len(df):,}")
    info("Band averages rows", f"{len(df_band):,}")
    info("File summary rows",  f"{len(df_fs):,}")
    for p in [csv_main,csv_band,csv_cx,csv_sum]:
        if p.exists(): info(p.name, f"{p.stat().st_size:,} bytes")
summary()


  [PASS]  blasting_all_results.csv exists  → D:\1 placement\IAESTE INTERNSHIP CZECH\iaeste26-blasting-sound-main\iaeste26-blasting-sound-main\results\blasting_all_results.csv
  [PASS]  filter_complexity.csv exists  → D:\1 placement\IAESTE INTERNSHIP CZECH\iaeste26-blasting-sound-main\iaeste26-blasting-sound-main\results\filter_complexity.csv
  [PASS]  blasting_band_averages.csv written  → 56,483 bytes
  [PASS]  blasting_file_summary.csv written  → 4,041 bytes


  [PASS]  Reloaded row count matches original  → 20025 rows
  [PASS]  All parameter columns present after reload
  [PASS]  No NaN in RMS after reload
  [PASS]  filename column present
  [PASS]  sensor column present
  [INFO]  Full results rows: 20,025
  [INFO]  Band averages rows: 450
  [INFO]  File summary rows: 30
  [INFO]  blasting_all_results.csv: 3,718,200 bytes
  [INFO]  blasting_band_averages.csv: 56,483 bytes
  [INFO]  filter_complexity.csv: 826 bytes
  [INFO]  blasting_file_summary.csv: 4,041 bytes

  PASS: 9/9  |  FAIL: 0/9
  Score: 100%
